# Scientific Publication Graphing Workspace: Interactive Research Notebook

This interactive notebook demonstrates the publication-quality plotting workflow across:
1. **DFT Electronic Structure**: Total & Projected DOS, and Electronic Band Structure
2. **Adsorption Energy Analysis**: Grouped bar charts with uncertainty and scaling relations
3. **Heterogeneous Catalysis**: Free energy reaction pathways and Sabatier volcano curves
4. **Chemical Engineering Kinetics**: Arrhenius regression diagnostics and reactor performance curves
5. **Quantum ESPRESSO Integration**: Direct parsing and publication visualization of pw.x, dos.x, projwfc.x, bands.x, and neb.x

In [1]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.config import EXAMPLES_DATA_DIR, FIGURES_DFT_DIR, FIGURES_CATALYSIS_DIR, FIGURES_CHEMENG_DIR
from scripts.utils import apply_publication_style, create_publication_figure, save_publication_figure, load_csv_data
from modules.dft.electronic_structure import plot_dos, plot_pdos, plot_band_structure
from modules.dft.adsorption import plot_adsorption_energies, plot_adsorption_scaling
from modules.catalysis.reaction_energy import plot_reaction_pathway
from modules.catalysis.volcano import plot_volcano
from modules.chemical_engineering.kinetics import plot_arrhenius, plot_conversion_selectivity, plot_reactor_profiles
from parsers.quantum_espresso import (
    parse_pw_output,
    parse_dos_file,
    QEPdosCollection,
    parse_bands_gnu,
    parse_neb_output,
    plot_qe_dos,
    plot_qe_pdos,
    plot_qe_bands,
    plot_qe_neb_profile,
)

# Apply publication style
apply_publication_style()

C:\Users\mjram\AppData\Local\Temp\ipykernel_8736\129590334.py:10: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 1. DFT Electronic Structure: DOS & Band Structure

In [2]:
# Load synthetic DOS benchmark
df_dos = load_csv_data(EXAMPLES_DATA_DIR / 'dft_dos_synthetic.csv')

fig, (ax_tdos, ax_pdos) = create_publication_figure(
    columns='single',
    height_mm=110.0,
    nrows=2,
    ncols=1,
    sharex=True,
    gridspec_kw={'hspace': 0.08}
)

plot_dos(
    energies=df_dos['energy'],
    dos_up=df_dos['dos_spin_up'],
    dos_down=df_dos['dos_spin_down'],
    e_fermi=0.0,
    ax=ax_tdos,
    energy_range=(-7.0, 5.0),
    label_up=r'Total $\uparrow$',
    label_down=r'Total $\downarrow$'
)
ax_tdos.set_xlabel('')

pdos_dict = {
    r'Pt $5d$': {'up': df_dos['pt_5d_up'], 'down': df_dos['pt_5d_down']},
    r'O $2p$': {'up': df_dos['o_2p_up'], 'down': df_dos['o_2p_down']},
}
plot_pdos(
    energies=df_dos['energy'],
    projections=pdos_dict,
    total_dos=df_dos['total_dos'],
    e_fermi=0.0,
    ax=ax_pdos,
    energy_range=(-7.0, 5.0)
)
plt.show()

C:\Users\mjram\AppData\Local\Temp\ipykernel_8736\1470145740.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Adsorption Energy Comparisons

In [3]:
df_ads = load_csv_data(EXAMPLES_DATA_DIR / 'adsorption_energies_synthetic.csv')

fig, ax = create_publication_figure(columns='single', aspect_ratio='1.3')
plot_adsorption_energies(
    data=df_ads,
    surface_col='surface',
    adsorption_col='e_ads',
    site_col='site',
    error_col='error',
    ax=ax,
    reference_note='Ref: CO(g), Exothermic < 0'
)
plt.show()

C:\Users\mjram\AppData\Local\Temp\ipykernel_8736\1627181927.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Catalytic Reaction Pathways & Volcano Plots

In [4]:
df_rxn = load_csv_data(EXAMPLES_DATA_DIR / 'catalysis_reaction_profile_synthetic.csv')
steps = df_rxn['step_name'].tolist()
is_ts = df_rxn['is_transition_state'].tolist()
catalysts = [c for c in df_rxn.columns if c not in ('step_name', 'is_transition_state')]

pathways = {c: df_rxn[c].tolist() for c in catalysts}
ts_flags = {c: is_ts for c in catalysts}

fig, ax = create_publication_figure(columns='double', aspect_ratio='2.2')
plot_reaction_pathway(
    pathways=pathways,
    step_labels=steps,
    transition_states=ts_flags,
    ax=ax,
    title='CO Oxidation Reaction Free Energy Pathway (298.15 K, 1 bar)'
)
plt.show()

C:\Users\mjram\AppData\Local\Temp\ipykernel_8736\3259910289.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Chemical Engineering: Arrhenius & Reactor Profiles

In [5]:
df_arrh = load_csv_data(EXAMPLES_DATA_DIR / 'kinetics_arrhenius_synthetic.csv')
fig, fit_results = plot_arrhenius(
    temperatures_k=df_arrh['temperature_K'],
    rate_constants=df_arrh['rate_constant_k'],
    rate_constant_units=r'\mathrm{s}^{-1}',
    show_residuals=True,
    title='Arrhenius Kinetics with Residual Diagnostics'
)
plt.show()
print('Fitted Activation Energy:', round(fit_results['ea_kj_mol'], 2), '+/-', round(fit_results['ea_stderr_kj_mol'], 2), 'kJ/mol')
print('Fitted ln(A):', round(fit_results['ln_a'], 2), 'R^2:', round(fit_results['r_squared'], 4))

Fitted Activation Energy: 82.66 +/- 0.55 kJ/mol
Fitted ln(A): 18.24 R^2: 0.9997


C:\Users\mjram\AppData\Local\Temp\ipykernel_8736\834088496.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Direct Quantum ESPRESSO Processing & Visualization

In [6]:
fixture_dir = project_root / 'tests' / 'fixtures' / 'qe'

# 5.1 Parse SCF output from pw.x
scf_out = parse_pw_output(fixture_dir / 'scf_spin_converged.out', strict=True)
print(f"Calculation: {scf_out.calculation_type}, Job Done: {scf_out.is_job_done}")
print(f"Total Energy: {scf_out.total_energy_ry:.6f} Ry = {scf_out.total_energy_ev:.4f} eV")
print(f"Fermi Energy: {scf_out.fermi_energy} eV")

# 5.2 Parse and plot spin-polarized DOS from dos.x
dos_data = parse_dos_file(fixture_dir / 'dos_spin.dat', e_fermi=scf_out.fermi_energy)
fig, ax = create_publication_figure(columns='single', aspect_ratio='golden')
plot_qe_dos(dos_data, ax=ax, energy_range=(-4.0, 4.0))
plt.show()

# 5.3 Parse and plot NEB reaction barrier from neb.x
neb_data = parse_neb_output(fixture_dir / 'neb.dat')
fig, ax = create_publication_figure(columns='single', aspect_ratio='golden')
plot_qe_neb_profile(neb_data, ax=ax, title='Quantum ESPRESSO NEB Reaction Profile')
plt.show()
print(f"Forward Activation Barrier: {neb_data.forward_barrier_ev:.2f} eV")
print(f"Reverse Activation Barrier: {neb_data.reverse_barrier_ev:.2f} eV")

C:\Users\mjram\AppData\Local\Temp\ipykernel_8736\3618547014.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Calculation: scf, Job Done: True
Total Energy: -62.152431 Ry = -845.6269 eV
Fermi Energy: 12.3456 eV
Forward Activation Barrier: 0.85 eV
Reverse Activation Barrier: 1.50 eV


C:\Users\mjram\AppData\Local\Temp\ipykernel_8736\3618547014.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
